# Approved Invalid-Pair Exclusion

Builds a new cleaned dataset outside `original/` by excluding exactly the approved invalid image-label pairs in the validation report. The original dataset remains unchanged. Valid background images with empty labels are retained.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from weapon_threat_detection.artifacts import configure_logger, utc_timestamp, write_json
from weapon_threat_detection.config import load_config
from weapon_threat_detection.preparation import materialize_clean_dataset, write_removal_report
from weapon_threat_detection.validation import (
    build_dataset_statistics,
    build_integrity_report,
    summarize_records,
    validate_dataset,
    write_validation_csv,
)

config = load_config(ROOT / 'configs' / 'project.yaml')
source = ROOT / config['project']['original_data_dir']
target = ROOT / config['project']['processed_data_dir'] / 'cleaned_v1'
reports_dir = ROOT / config['project']['reports_dir']
validation_csv = max(reports_dir.glob('validation_records_*.csv'), key=lambda path: path.stat().st_mtime)
logger = configure_logger(ROOT / config['project']['logs_dir'], 'approved_pair_exclusion')
removals = materialize_clean_dataset(
    source,
    target,
    validation_csv,
    config['dataset']['splits'],
    config['dataset']['class_names'],
    config['preparation']['mode'],
)
stamp = utc_timestamp()
removal_report = write_removal_report(removals, reports_dir / f'removal_report_{stamp}.csv')
final_records = validate_dataset(target, config['dataset']['splits'], len(config['dataset']['class_names']), config['dataset']['image_extensions'])
final_validation = write_validation_csv(final_records, reports_dir / f'final_validation_records_{stamp}.csv')
final_integrity = write_json(reports_dir / f'final_integrity_report_{stamp}.json', build_integrity_report(final_records, config['dataset']['splits']))
final_statistics = write_json(reports_dir / f'final_dataset_statistics_{stamp}.json', build_dataset_statistics(final_records, config['dataset']['class_names']))
summary = summarize_records(final_records)
if any(status not in {'valid', 'valid_background'} for status in summary):
    raise RuntimeError(f'Cleaned dataset still has validation errors: {summary}')
logger.info('Excluded %s approved invalid pairs without modifying original/', len(removals))
logger.info('Final validation summary: %s', summary)
print(f'Excluded pairs: {len(removals)}')
print(f'Removal report: {removal_report}')
print(f'Final validation records: {final_validation}')
print(f'Final integrity report: {final_integrity}')
print(f'Final statistics: {final_statistics}')
print(summary)

Excluded pairs: 518
Removal report: /Users/mohamedtarek/weapon130/reports/removal_report_20260723T090604Z.csv
Final validation records: /Users/mohamedtarek/weapon130/reports/final_validation_records_20260723T090604Z.csv
Final integrity report: /Users/mohamedtarek/weapon130/reports/final_integrity_report_20260723T090604Z.json
Final statistics: /Users/mohamedtarek/weapon130/reports/final_dataset_statistics_20260723T090604Z.json
{'valid': 117903, 'valid_background': 12342}
